In [2]:
import re
import numpy as np
import pandas as pd

In [3]:
def normalize_series(s):
    """Lowercase and strip while preserving NaN values."""
    return s.where(s.isna(), s.astype(str).str.lower().str.strip())

In [6]:

def clean_ebay_to_auto24_format(input_csv_path, output_csv_path=None):
    print(f"Reading eBay data from: {input_csv_path}")
    df = pd.read_csv(input_csv_path, encoding="latin-1", low_memory=False)
    out = pd.DataFrame()
    # 1. Make & Model (normalized lowercase, spaces instead of underscores)
    out["make"] = (
        normalize_series(df["brand"])
        .str.replace("_", " ", regex=False)
        .replace({"sonstige autos": "other"})
    )
    out["model"] = normalize_series(df["model_final"])
    # 2. Numeric features & target
    
    out["price_eur"] = pd.to_numeric(df["price"], errors="coerce")
    out["log_price"] = np.log1p(out["price_eur"])
    out["mileage_km"] = pd.to_numeric(df["kilometer"], errors="coerce")
    out["power_ps"] = pd.to_numeric(df["powerPS"], errors="coerce")
    out["power_kw"] = (out["power_ps"] / 1.35962).round(1)
    out["car_age"] = pd.to_numeric(df["car_age"], errors="coerce")
    # 3. Fuel type (German -> English)
    fuel_map = {
        "benzin": "petrol",
        "diesel": "diesel",
        "hybrid": "hybrid",
        "elektro": "electric",
        "lpg": "lpg",
        "cng": "cng",
        "andere": "other",
    }
    out["fuel_type"] = normalize_series(df["fuelType"]).map(fuel_map)
    # 4. Transmission (German -> English)
    transmission_map = {
        "manuell": "manual",
        "automatik": "automatic",
    }
    out["transmission"] = normalize_series(df["gearbox"]).map(transmission_map)
    # 5. Body type (German -> English)
    body_map = {
        "limousine": "sedan",
        "kombi": "wagon",
        "kleinwagen": "hatchback",
        "bus": "minivan",
        "suv": "suv",
        "coupe": "coupe",
        "cabrio": "cabrio",
        "andere": "other",
    }
    out["body_type"] = normalize_series(df["vehicleType"]).map(body_map)
    # 6. Damaged status (ja/nein -> True/False, preserving NaN)
    damage_map = {"ja": True, "nein": False}
    out["damaged"] = normalize_series(df["notRepairedDamage"]).map(damage_map)
    # 7. Drop invalid prices if any exist
    out = out.dropna(subset=["price_eur"]).copy()
    out = out[out["price_eur"] > 100].copy()
    if output_csv_path:
        out.to_csv(output_csv_path, index=False)
        print(f"Saved English/Auto24-formatted eBay CSV to: {output_csv_path} (Rows: {len(out)})")
    return out



In [7]:
# Example Usage:
ebay_in = r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\ebay\autos_clean.csv"
ebay_out = r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\ebay\ebay_cleaned_english_auto24_format.csv"
df_ebay_english = clean_ebay_to_auto24_format(ebay_in, ebay_out)
print(df_ebay_english.head())

Reading eBay data from: c:\Users\Andi\learn_projects\GIT\UsedCarPrice\ebay\autos_clean.csv
Saved English/Auto24-formatted eBay CSV to: c:\Users\Andi\learn_projects\GIT\UsedCarPrice\ebay\ebay_cleaned_english_auto24_format.csv (Rows: 238272)
         make           model  price_eur  log_price  mileage_km  power_ps  \
0        jeep  grand cherokee       9800   9.190240      125000       163   
1  volkswagen            golf       1500   7.313887      150000        75   
2       skoda           fabia       3600   8.188967       90000        69   
3         bmw            316i        650   6.478510      150000       102   
4     peugeot             206       2200   7.696667      150000       109   

   power_kw  car_age fuel_type transmission  body_type damaged  
0     119.9       22    diesel    automatic        suv     NaN  
1      55.2       25    petrol       manual  hatchback   False  
2      50.7       18    diesel       manual  hatchback   False  
3      75.0       31    petrol       

In [ ]:
auto24 = pd.read_csv(r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\auto24\Data Backup\auto24_vehicles_nov14_week1.csv")
# 1. Keep only passenger cars (drop boats, trailers, tractors, motorcycles)
car_bodies = {"cabrio", "coupe", "hatchback", "minivan", "pickup", "sedan", "wagon", "van"}
auto24 = auto24[auto24["body_type"].isin(car_bodies)].copy()
# 2. Normalize make & model to match eBay's lowercase formatting
auto24["make"] = (
    auto24["make"].str.lower().str.strip()
    .str.replace("-", " ", regex=False)
    .replace({"mercedes amg": "mercedes benz"})
)
auto24["model"] = (
    auto24["model"].astype(str).str.lower().str.strip()
    .str.replace(r"[-_/]+", " ", regex=True)
    .str.replace(r"\b([a-z])\s+(\d+)\b", r"\1\2", regex=True)
)
# 3. Collapse Auto24-only categories into eBay's English categories
auto24["fuel_type"] = auto24["fuel_type"].replace({"plug_in_hybrid": "hybrid", "hydrogen": "other"})
auto24["transmission"] = auto24["transmission"].replace({"semi_automatic": "automatic"})
auto24["body_type"] = auto24["body_type"].replace({"van": "minivan", "pickup": "suv"})